# Chapter 8: Deep Learning — Many Layers, CNNs, Why Depth Matters

"Deep" learning just means neural networks with **many layers**.

```
Shallow: Input → [Hidden] → Output          (1 hidden layer)
Deep:    Input → [H1] → [H2] → [H3] → ...  → Output  (many layers)
```

Why depth matters: each layer learns increasingly **abstract** features.

```
Image recognition example:
  Layer 1: edges, colors
  Layer 2: textures, corners
  Layer 3: eyes, ears, wheels
  Layer 4: faces, cars, animals
```

The deeper you go, the more **complex concepts** the network can represent.

---
## Depth vs Width

Two ways to add capacity to a network:
- **Wider**: more neurons per layer (brute force)
- **Deeper**: more layers (hierarchical abstraction)

Deep networks are more **parameter-efficient** — they reuse lower-level features.

In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

def count_params(architecture):
    """Count total parameters for a given architecture."""
    total = 0
    for i in range(len(architecture) - 1):
        weights = architecture[i] * architecture[i+1]
        biases = architecture[i+1]
        total += weights + biases
    return total

# Compare architectures
architectures = {
    "Shallow wide":    [10, 100, 1],
    "Medium":          [10, 32, 16, 1],
    "Deep narrow":     [10, 16, 16, 8, 4, 1],
    "Very deep":       [10, 16, 16, 16, 16, 8, 4, 1],
}

print("Architecture Comparison:")
print(f"{'Name':<18} {'Shape':<28} {'Parameters':>12} {'Layers':>8}")
print(f"{'─'*18} {'─'*28} {'─'*12} {'─'*8}")

for name, arch in architectures.items():
    params = count_params(arch)
    layers = len(arch) - 2  # hidden layers
    shape = "→".join(str(a) for a in arch)
    print(f"{name:<18} {shape:<28} {params:>12,} {layers:>8}")

print("\nDeep narrow uses FEWER parameters but can learn MORE complex patterns.")
print("Each layer builds on the previous layer's abstractions.")

---
## What Each Layer Learns

Let's demonstrate with a concrete example: learning to detect patterns in numbers.

In [ ]:
# Demonstrate hierarchical feature learning
# Task: classify numbers into categories based on multiple properties
np.random.seed(42)

# Generate data: [value, is_even (noisy), magnitude_category]
N = 200
values = np.random.randn(N, 4)  # 4 raw features

# Target: complex non-linear pattern
# Class 1 if (feature1 > 0 AND feature2 > 0) OR (feature3 < 0 AND feature4 < 0)
targets = (((values[:, 0] > 0) & (values[:, 1] > 0)) | 
           ((values[:, 2] < 0) & (values[:, 3] < 0))).astype(float).reshape(-1, 1)

# Train shallow vs deep
def train_network(X, y, hidden_sizes, epochs=2000, lr=0.1):
    layers = [X.shape[1]] + hidden_sizes + [1]
    weights = []
    biases = []
    for i in range(len(layers) - 1):
        w = np.random.randn(layers[i], layers[i+1]) * np.sqrt(2.0 / layers[i])
        b = np.zeros((1, layers[i+1]))
        weights.append(w)
        biases.append(b)
    
    for epoch in range(epochs):
        # Forward
        activations = [X]
        for i in range(len(weights)):
            z = activations[-1] @ weights[i] + biases[i]
            if i < len(weights) - 1:
                a = np.maximum(0, z)  # ReLU
            else:
                a = sigmoid(z)  # Output sigmoid
            activations.append(a)
        
        # Loss
        output = activations[-1]
        loss = -np.mean(y * np.log(output + 1e-8) + (1-y) * np.log(1 - output + 1e-8))
        
        # Backward
        delta = output - y
        for i in range(len(weights) - 1, -1, -1):
            dw = activations[i].T @ delta / N
            db = np.sum(delta, axis=0, keepdims=True) / N
            if i > 0:
                delta = (delta @ weights[i].T) * (activations[i] > 0)
            weights[i] -= lr * dw
            biases[i] -= lr * db
    
    preds = (output > 0.5).astype(float)
    accuracy = np.mean(preds == y)
    return accuracy, count_params(layers)

configs = [
    ("No hidden layer",  []),
    ("1 layer (wide)",   [32]),
    ("2 layers",         [16, 8]),
    ("3 layers",         [12, 8, 4]),
    ("4 layers",         [8, 8, 4, 4]),
]

print("Depth vs Accuracy on Complex Pattern:\n")
print(f"{'Architecture':<20} {'Params':>8} {'Accuracy':>10}")
print(f"{'─'*20} {'─'*8} {'─'*10}")

for name, hidden in configs:
    acc, params = train_network(values, targets, hidden)
    bar = "█" * int(acc * 30)
    print(f"{name:<20} {params:>8} {acc:>10.1%}  {bar}")

print("\nDeeper networks handle complex patterns better.")
print("Layer 1 learns simple conditions (x>0).")
print("Layer 2 combines them (x>0 AND y>0).")
print("Layer 3 combines combinations (pattern1 OR pattern2).")

---
## Convolutional Neural Networks (CNNs)

Regular networks treat every input independently. But for images, **nearby pixels are related**.

A CNN uses **filters** (small weight matrices) that slide across the input, detecting local patterns.

```
Image (5×5)              Filter (3×3)         Feature Map
┌─────────────┐          ┌───────┐           ┌─────────┐
│ 0 1 1 0 0   │          │ 1 0 1 │           │ ? ? ?   │
│ 0 1 1 0 0   │    *     │ 0 1 0 │     =     │ ? ? ?   │
│ 0 1 1 0 0   │          │ 1 0 1 │           │ ? ? ?   │
│ 0 0 0 0 0   │          └───────┘           └─────���───┘
│ 0 0 0 0 0   │
���───────���─────┘
```

The filter slides across the image. At each position, it computes a dot product (element-wise multiply and sum).

In [ ]:
# Manual convolution to show how it works

# A simple 7×7 "image" with a vertical line
image = np.array([
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0],
])

# Vertical edge detector filter
vertical_filter = np.array([
    [-1, 0, 1],
    [-1, 0, 1],
    [-1, 0, 1],
])

# Horizontal edge detector filter
horizontal_filter = np.array([
    [-1, -1, -1],
    [ 0,  0,  0],
    [ 1,  1,  1],
])

def convolve2d(image, kernel):
    h, w = image.shape
    kh, kw = kernel.shape
    out_h = h - kh + 1
    out_w = w - kw + 1
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = image[i:i+kh, j:j+kw]
            output[i, j] = np.sum(patch * kernel)
    return output

print("Input image (vertical line):")
for row in image:
    print("  " + " ".join("█" if v else "·" for v in row))

print("\nVertical edge filter:")
for row in vertical_filter:
    print("  " + " ".join(f"{v:+d}" for v in row))

result_v = convolve2d(image, vertical_filter)
print("\nResult (vertical filter → detects vertical edges):")
for row in result_v:
    print("  " + " ".join(f"{int(v):+d}" if v != 0 else " 0" for v in row))

print("\n  Negative values = left edge, Positive = right edge")
print("  The filter FOUND the vertical line!")

result_h = convolve2d(image, horizontal_filter)
print("\nResult (horizontal filter):")
for row in result_h:
    print("  " + " ".join(f"{int(v):+d}" if v != 0 else " 0" for v in row))
print("\n  All zeros! No horizontal edges in a vertical line.")

---
## CNN Architecture: Conv → Pool → Conv → Pool → Dense

A typical CNN stacks:

```
1. Convolution layer  → detect features (edges, textures)
2. Pooling layer      → shrink the image (keep important info)
3. Repeat 1-2         → detect higher-level features
4. Flatten             → convert 2D to 1D
5. Dense layers        → classify based on detected features
```

**Pooling** (usually max pooling) reduces size by keeping the maximum value in each region:

```
┌─────┬─────┐         ┌─────┐
│ 1 3 │ 2 1 │         │ 3 2 │
│ 0 2 │ 0 2 │   →     │ 4 5 │
├─────┼─────┤  max     └─────┘
│ 4 1 │ 5 0 │  pool
│ 3 0 │ 1 1 │  (2×2)
└─────┴─────┘
```

In [ ]:
def max_pool_2d(feature_map, pool_size=2):
    h, w = feature_map.shape
    out_h = h // pool_size
    out_w = w // pool_size
    output = np.zeros((out_h, out_w))
    for i in range(out_h):
        for j in range(out_w):
            patch = feature_map[i*pool_size:(i+1)*pool_size, j*pool_size:(j+1)*pool_size]
            output[i, j] = np.max(patch)
    return output

# Full CNN-like pipeline on our simple image
print("CNN Pipeline on 7×7 Image:\n")

# Step 1: Convolution
print("1. Convolution (3×3 vertical filter):")
conv_result = convolve2d(image, vertical_filter)
print(f"   Input: {image.shape} → Output: {conv_result.shape}")
for row in conv_result:
    print("   " + " ".join(f"{int(v):>2}" for v in row))

# Step 2: ReLU activation
print("\n2. ReLU activation (remove negatives):")
relu_result = np.maximum(0, conv_result)
for row in relu_result:
    print("   " + " ".join(f"{int(v):>2}" for v in row))

# Step 3: Max pooling
print("\n3. Max pooling (2×2, keeps strongest signals):")
# Trim to even size first
trimmed = relu_result[:4, :4]
pool_result = max_pool_2d(trimmed)
print(f"   {trimmed.shape} → {pool_result.shape}")
for row in pool_result:
    print("   " + " ".join(f"{int(v):>2}" for v in row))

# Step 4: Flatten
flat = pool_result.flatten()
print(f"\n4. Flatten: {pool_result.shape} → [{len(flat)} values]")
print(f"   {flat}")

print("\n5. Dense layer would classify based on these features.")
print("\nThe CNN reduced 7×7=49 pixels to 4 meaningful features!")

---
## Why CNNs Work: Key Ideas

Three properties make CNNs efficient for spatial data:

In [ ]:
print("Why CNNs Are Powerful:\n")

print("1. PARAMETER SHARING")
print("   Regular network: each pixel gets its own weight")
print("   CNN: one filter is reused across the entire image")
print(f"   Example: 28×28 image, 3×3 filter")
print(f"   Regular: 28×28 = 784 weights per neuron")
print(f"   CNN:     3×3  = 9 weights per filter (reused everywhere!)")
print()

print("2. TRANSLATION INVARIANCE")
print("   A cat in the top-left corner is detected the same")
print("   as a cat in the bottom-right corner.")
print("   The filter slides everywhere, looking for the same pattern.")
print()

print("3. HIERARCHICAL FEATURES")
print("   Layer 1 filters: ─ │ ╱ ╲ (edges)")
print("   Layer 2 filters: ┌┐ ○ △ (shapes from edges)")
print("   Layer 3 filters: 👁 🐾 🚗 (objects from shapes)")
print("   Each layer combines previous layer's features!")
print()

# Parameter comparison
image_size = 28 * 28  # MNIST digit
dense_params = image_size * 128 + 128  # one dense layer with 128 neurons
conv_params = 3 * 3 * 32 + 32  # 32 filters of 3×3

print("Parameter Count (28×28 MNIST digit):")
print(f"  Dense layer (128 neurons):    {dense_params:>8,} parameters")
print(f"  Conv layer (32 3×3 filters):  {conv_params:>8,} parameters")
print(f"  CNN uses {dense_params/conv_params:.0f}× fewer parameters!")

---
## Practical CNN: Digit Recognition with PyTorch

Let's see a real CNN in action (just the architecture — training takes GPU time).

In [ ]:
# CNN architecture for MNIST (pseudo-code style, no training)
print("Typical CNN for Handwritten Digit Recognition (MNIST):\n")

layers = [
    ("Input",        "28 × 28 × 1",    "—",       "grayscale image"),
    ("Conv2d(32)",   "26 × 26 × 32",   "320",     "32 filters detect 32 edge types"),
    ("ReLU",         "26 × 26 × 32",   "0",       "remove negatives"),
    ("MaxPool(2×2)", "13 × 13 × 32",   "0",       "shrink by half"),
    ("Conv2d(64)",   "11 × 11 × 64",   "18,496",  "64 filters detect shapes"),
    ("ReLU",         "11 × 11 × 64",   "0",       "remove negatives"),
    ("MaxPool(2×2)", "5 × 5 × 64",     "0",       "shrink by half"),
    ("Flatten",      "1600",            "0",       "2D → 1D"),
    ("Dense(128)",   "128",             "204,928", "combine all features"),
    ("Dense(10)",    "10",              "1,290",   "one score per digit (0-9)"),
    ("Softmax",      "10",              "0",       "scores → probabilities"),
]

print(f"{'Layer':<16} {'Output Shape':<18} {'Params':>10} {'Purpose'}")
print(f"{'─'*16} {'─'*18} {'─'*10} {'─'*35}")
for name, shape, params, purpose in layers:
    print(f"{name:<16} {shape:<18} {params:>10} {purpose}")

print(f"\nTotal: ~225,000 parameters")
print(f"Accuracy on MNIST: ~99%")
print(f"\nA fully-connected network would need ~100,000 parameters")
print(f"just for the FIRST layer, and would perform worse.")

---
## Making Deep Networks Work: Key Techniques

Deep networks are powerful but hard to train. These techniques solved the problems:

In [ ]:
print("Techniques That Made Deep Learning Possible:\n")

techniques = [
    ("ReLU activation", "2012",
     "Replaces sigmoid in hidden layers",
     "Fixes vanishing gradient — derivative is 1, not 0.25"),
    ("Dropout", "2014",
     "Randomly disable neurons during training",
     "Prevents overfitting — forces redundant learning"),
    ("Batch Normalization", "2015",
     "Normalize inputs to each layer",
     "Stabilizes training, allows higher learning rates"),
    ("Residual Connections", "2015",
     "Skip connections: add input to output",
     "Enables 100+ layer networks (ResNet)"),
    ("Adam Optimizer", "2015",
     "Adaptive learning rate per parameter",
     "Faster convergence than basic gradient descent"),
]

for name, year, what, why in techniques:
    print(f"  {name} ({year})")
    print(f"    What: {what}")
    print(f"    Why:  {why}")
    print()

---
## Residual Connections: The Key to Very Deep Networks

Regular layer: `output = f(input)`

Residual layer: `output = f(input) + input`

The `+ input` part is a "skip connection." It means:
- The layer only needs to learn the **difference** (residual)
- Gradients flow directly through the skip connection
- Worst case: the layer learns nothing (output = input) → no harm done

This is critical for transformers too (Ch 12).

In [ ]:
# Demonstrate residual connections
np.random.seed(42)

def regular_forward(x, weights_list):
    """Regular: each layer transforms the input."""
    for W in weights_list:
        x = np.maximum(0, x @ W)  # ReLU
    return x

def residual_forward(x, weights_list):
    """Residual: each layer ADDS to the input."""
    for W in weights_list:
        residual = np.maximum(0, x @ W)  # what the layer learned
        x = x + residual  # add it to the original (skip connection)
    return x

# 10 layers, 4 neurons each
weights = [np.random.randn(4, 4) * 0.3 for _ in range(10)]
x = np.array([[1.0, 0.5, -0.3, 0.8]])

print("Signal strength through 10 layers:\n")
print(f"{'Layer':>6} {'Regular':>20} {'Residual':>20}")
print(f"{'─'*6} {'─'*20} {'─'*20}")

x_reg = x.copy()
x_res = x.copy()

for i, W in enumerate(weights):
    x_reg = np.maximum(0, x_reg @ W)
    x_res = x_res + np.maximum(0, x_res @ W)
    
    reg_norm = np.linalg.norm(x_reg)
    res_norm = np.linalg.norm(x_res)
    
    reg_bar = "█" * min(30, int(reg_norm * 10))
    res_bar = "█" * min(30, int(res_norm * 2))
    
    print(f"{i+1:>6} {reg_norm:>8.4f} {reg_bar:<11} {res_norm:>8.4f} {res_bar}")

print(f"\nRegular: signal may vanish or explode through many layers.")
print(f"Residual: signal stays alive because of the skip connection.")

---
## Dropout: Preventing Overfitting

During training, randomly set some neurons to 0. This forces the network to not rely on any single neuron.

```
Without dropout:  [0.5, 0.3, 0.8, 0.2, 0.9] → all neurons active
With dropout(50%): [0.5, 0.0, 0.8, 0.0, 0.9] → random neurons off
```

At test time: all neurons are active (no dropout).

In [ ]:
# Dropout visualization
np.random.seed(42)

layer_output = np.array([0.5, 0.3, 0.8, 0.2, 0.9, 0.1, 0.7, 0.4])
dropout_rate = 0.5

print("Dropout in Action (50% rate):\n")
print("Original layer output:")
for i, v in enumerate(layer_output):
    bar = "█" * int(v * 20)
    print(f"  Neuron {i}: {v:.1f}  {bar}")

print("\nWith dropout (3 different random masks):")
for trial in range(3):
    mask = np.random.binomial(1, 1 - dropout_rate, len(layer_output))
    dropped = layer_output * mask
    print(f"\n  Trial {trial + 1}: mask = {mask}")
    for i, (v, m) in enumerate(zip(dropped, mask)):
        if m:
            bar = "█" * int(v * 20)
            print(f"    Neuron {i}: {v:.1f}  {bar}")
        else:
            print(f"    Neuron {i}: OFF  ✗")

print("\nEach training step sees a different 'thinned' network.")
print("The network learns redundant representations → more robust.")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **Depth** | More layers = more abstract features |
| **Width vs Depth** | Deep networks are more parameter-efficient |
| **CNN** | Filters slide across input, detect local patterns |
| **Pooling** | Shrinks feature maps, keeps strongest signals |
| **Parameter sharing** | CNN filters reused everywhere → efficient |
| **Residual connections** | Skip connections enable very deep networks |
| **Dropout** | Random neuron disabling prevents overfitting |
| **Batch normalization** | Stabilizes training of deep networks |

**Next: Part 3 — Generative AI** (sequence models, transformers, and how LLMs work)